### Setup

In [ ]:
# Install dependencies
!pip install pyyaml pillow torchvision

import os
import zipfile
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# --- CONFIGURATION ---
ZIP_PATH = "/content/drive/MyDrive/split_centroid_dataset/split_centroid_dataset.zip"
LOCAL_ROOT = "/content/dataset"
# ---------------------

# Extract Dataset
if os.path.exists(ZIP_PATH):
    print("Extracting Centroid Dataset...")
    if not os.path.exists(LOCAL_ROOT):
        os.makedirs(LOCAL_ROOT)
    !unzip -qo {ZIP_PATH} -d {LOCAL_ROOT}
    print("Extraction complete.")
else:
    print(f"Error: {ZIP_PATH} not found.")

### Architecture and Preprocessing

**v3 = the STN-FOMO recipe, minus the STN.**

The original FOMO finetune uses MobileNetV2 **alpha=0.35** cut at block 7 (stride 8
-> 60x60, ~32 ch) with a tiny 1x1->1x1 head. STN-FOMO is a different, ~10x larger
network: **full-width MobileNetV2 (alpha=1.0)** cut at block 13 (stride 16 -> 30x30,
**96 ch**) with a bigger head (depthwise 3x3 -> 1x1 96->96 -> 1x1 96->C), plus a
Spatial Transformer.

In the trained STN-FOMO checkpoint the STN collapsed to a ~0 deg identity transform
(it predicts the same near-identity for every image), and an ablation that feeds the
backbone straight into the head changed detection scores by < 0.004. So the accuracy
margin comes from the **deeper backbone + bigger head + per-class sigmoid**, not the
rotation. v3 keeps exactly those parts and **drops the STN**.

This uses torchvision's ImageNet-pretrained alpha=1.0 weights directly, so no local
mobilenetv2_0.35 checkpoint / key-remapping is needed.

The dataset loader accepts **either** centroid labels (`cls x_c y_c`) **or** YOLO-OBB
labels (`cls x1 y1 x2 y2 x3 y3 x4 y4`); OBB boxes are reduced to their centroid (mean
of the 4 corners), so this notebook trains directly on the `obb_dataset`. Set
`DATASET_DIRNAME` (in the training cell) to your dataset folder; a train/val split or
a flat images/+labels/ layout both work.

In [ ]:
import math
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import MobileNet_V2_Weights
from PIL import Image
import glob
import yaml

# --- BACKBONE: full-width MobileNetV2 (alpha=1.0), ImageNet-pretrained ---
# STN-FOMO uses the full-width backbone (32-channel stem, 96 channels at block 13),
# i.e. width_mult=1.0 -- NOT the 0.35 of the original FOMO finetune. torchvision
# ships the ImageNet-pretrained alpha=1.0 weights, so we load them directly and
# keep only `.features` (the classifier and final 1x1 conv are unused by FOMO).
def load_mobilenet_v2_100():
    """Return MobileNetV2 alpha=1.0 `.features`, ImageNet-pretrained."""
    net = models.mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
    return net.features


# --- TARGET SMEARING (CenterNet-style soft Gaussian heatmaps) ---
# Identical to the FOMO finetune. Smearing is applied ONLY to the railroad-crossing
# class (id 0): it is the one class whose signs rotate, so the exact centre cell is
# ambiguous on tilted examples and a tolerant Gaussian target helps the model fire.
# Its centroid is splatted as a 2D Gaussian (peak 1.0 at the centre cell, decaying
# outward). The OTHER classes are static and keep a single hard cell. The model has
# NO background channel: one sigmoid heatmap per class, penalty-reduced focal loss.
# GAUSS_SIGMA is the smear width in GRID CELLS (~1 cell => bump spanning ~+/-3).
# Per-class focal loss hyperparameters.
# alpha controls the positive/negative balance per class:
#   high alpha -> model pushed to FIRE (fewer false negatives, more false positives)
#   low alpha  -> model pushed to SUPPRESS (fewer false positives, more false negatives)
#
# railroad-crossing: 0.75 — sparse, rotating, hard to hit; prioritise recall.
# lights-on:         0.50 — visually similar to lights-off; α=0.25 caused 13% FNs
#                           confused as lights-off, so we raise it to balance.
# lights-off:        0.25 — achieves 100% recall with 0.25; keep conservative.
# trefolo:           0.25 — achieves ~100% recall with 0.25; keep conservative.
FOCAL_GAMMA = 2.0
CLASS_ALPHA = {
    "railroad-crossing": 0.75,
    "lights-on":         0.50,
    "lights-off":        0.25,
    "trefolo":           0.25,
}

# Weight of the localization auxiliary loss (loc_fc branch).
LOC_WEIGHT = 0.1

RAILROAD_CROSSING_ID = 0


def per_cell_focal_loss(logits, target, alpha, gamma=FOCAL_GAMMA):
    """RetinaNet focal loss treating every grid cell as an independent binary classifier.

    Identical to train_stn_fomo.py. logits, target: (B, C, H, W).
    alpha: (C,) per-class positive-weight tensor on the same device.
    Normalized by the number of positive cells (RetinaNet-style).
    """
    prob = torch.sigmoid(logits)
    ce   = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    p_t     = prob * target + (1.0 - prob) * (1.0 - target)
    alpha_t = alpha.view(1, -1, 1, 1) * target + (1.0 - alpha.view(1, -1, 1, 1)) * (1.0 - target)
    loss    = alpha_t * (1.0 - p_t).pow(gamma) * ce
    num_pos = target.eq(1.0).sum().clamp(min=1.0)
    return loss.sum() / num_pos


def build_class_alpha(class_names):
    """Per-class focal alpha from CLASS_ALPHA table. Unknown classes default to 0.25."""
    alpha = [CLASS_ALPHA.get(n, 0.25) for n in class_names]
    return torch.tensor(alpha, dtype=torch.float32)


class FOMO_V3_480(nn.Module):
    """STN-FOMO architecture without the Spatial Transformer.

    Full-width MobileNetV2 (alpha=1.0) cut at block 13 -> 1/16 resolution
    (480 -> 30) with 96 channels, then STN-FOMO's head: depthwise 3x3 -> ReLU ->
    1x1 (96->96) -> ReLU -> 1x1 (96->num_classes). One heatmap channel PER CLASS
    (no background); returns raw logits (sigmoid is applied in the loss / at decode).
    """
    def __init__(self, num_classes):
        super(FOMO_V3_480, self).__init__()
        backbone = load_mobilenet_v2_100()

        # Cut at block_13 for 1/16 resolution (480 -> 30); 96 channels at alpha=1.0.
        # (block_7 would give 1/8 -> 60, which is the original alpha=0.35 FOMO.)
        self.features = nn.Sequential(*list(backbone.children())[:14])

        # Infer the cut's output channels (96 here) instead of hardcoding, so the
        # head stays correct if the cut point / alpha ever changes.
        backbone_channels = self._infer_backbone_channels()

        # STN-FOMO head, minus the STN that used to sit in front of it.
        self.head = nn.Sequential(
            # depthwise 3x3 spatial mixing (groups == channels)
            nn.Conv2d(backbone_channels, backbone_channels, kernel_size=3,
                      padding=1, groups=backbone_channels),
            nn.ReLU(),
            # pointwise 1x1 cross-channel mixing
            nn.Conv2d(backbone_channels, backbone_channels, kernel_size=1),
            nn.ReLU(),
            # pointwise 1x1 classifier -> one logit heatmap per class
            nn.Conv2d(backbone_channels, num_classes, kernel_size=1),
        )

        # Localization auxiliary branch — mirrors STN-FOMO's loc_fc.
        # Reads backbone features via global-avg-pool then two FC layers, creating
        # a second gradient path into the backbone through rotation supervision.
        # Supervised during training only; not exported to ONNX (eval forward
        # returns only the detection output).
        self.loc_fc = nn.Sequential(
            nn.Linear(backbone_channels, 32),
            nn.ReLU(),
            nn.Linear(32, 2),  # predicts (cos θ, sin θ) of the crossing rotation
        )

    def _infer_backbone_channels(self):
        was_training = self.training
        self.eval()
        with torch.no_grad():
            channels = self.features(torch.zeros(1, 3, 64, 64)).shape[1]
        self.train(was_training)
        return channels

    def forward(self, x):
        feat = self.features(x)
        det = self.head(feat)
        if self.training:
            # Global-avg-pool -> loc_fc: the second gradient path into the backbone.
            gap = feat.mean(dim=(2, 3))  # (B, C)
            loc = self.loc_fc(gap)       # (B, 2) = (cos θ, sin θ)
            return det, loc
        return det

class YOLOCentroidDataset(Dataset):
    # grid_size defaults to 30 (stride 16) for v3; the alpha=0.35 FOMO used 60.
    def __init__(self, split_root, num_classes, img_size=480, grid_size=30):
        # Look for images and labels in the unzipped folder
        self.img_dir = os.path.join(split_root, "images")
        self.label_dir = os.path.join(split_root, "labels")
        self.img_paths = sorted([p for p in glob.glob(os.path.join(self.img_dir, "*"))
                                  if p.lower().endswith(('.jpg', '.jpeg', '.png'))])
        self.num_classes = num_classes
        self.grid_size = grid_size
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        img = Image.open(img_path).convert('RGB')
        label_path = os.path.join(self.label_dir, os.path.splitext(os.path.basename(img_path))[0] + ".txt")

        # Target = one heatmap per class (C, grid, grid), no bg channel.
        # ALL classes use a HARD single-cell target (value 1.0 at the centroid
        # cell, 0 elsewhere) — identical to train_stn_fomo.py. No Gaussian smearing.
        # The per-class focal alpha (0.75 for crossing, 0.25 for statics) handles
        # class imbalance without needing soft targets.
        #
        # Labels may be EITHER centroid format ("cls x_c y_c") OR YOLO-OBB
        # ("cls x1 y1 x2 y2 x3 y3 x4 y4"); OBB boxes are reduced to their centroid
        # (mean of the 4 corners) below, so the same loader trains on both.
        gs = self.grid_size
        target = torch.zeros((self.num_classes, gs, gs), dtype=torch.float32)
        # Rotation target for the localization branch: (cos θ, sin θ) of the
        # railroad-crossing barrier's long axis, extracted from OBB corners.
        # Defaults to (1, 0) = 0° when no crossing is present in the image.
        loc_target     = torch.tensor([1.0, 0.0], dtype=torch.float32)
        has_crossing   = torch.tensor(0.0,         dtype=torch.float32)
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.split()
                    if len(parts) < 3: continue
                    cls_id = int(parts[0])
                    if not (0 <= cls_id < self.num_classes): continue
                    coords = [float(v) for v in parts[1:]]
                    if len(coords) >= 8:
                        # YOLO-OBB: cls x1 y1 x2 y2 x3 y3 x4 y4
                        # -> centroid = mean of the 4 corners (true centre of a
                        # rotated box, so it stays stable as the crossing tilts).
                        xs, ys = coords[0:8:2], coords[1:8:2]
                        x_c, y_c = sum(xs) / 4.0, sum(ys) / 4.0
                        # For railroad-crossing: extract rotation from the long edge.
                        if cls_id == RAILROAD_CROSSING_ID:
                            x1,y1,x2,y2,x3,y3 = coords[:6]
                            dx1,dy1 = x2-x1, y2-y1
                            dx2,dy2 = x3-x2, y3-y2
                            angle = math.atan2(dy1,dx1) if (dx1**2+dy1**2)>=(dx2**2+dy2**2) else math.atan2(dy2,dx2)
                            loc_target   = torch.tensor([math.cos(angle), math.sin(angle)], dtype=torch.float32)
                            has_crossing = torch.tensor(1.0, dtype=torch.float32)
                    else:
                        # centroid format: cls x_c y_c
                        x_c, y_c = coords[0], coords[1]
                    # Some OBB corners sit slightly outside [0,1]; clamp before binning.
                    x_c = min(max(x_c, 0.0), 0.999999)
                    y_c = min(max(y_c, 0.0), 0.999999)
                    gx = min(int(x_c * gs), gs - 1)
                    gy = min(int(y_c * gs), gs - 1)
                    # Hard single-cell target for every class — no smearing.
                    target[cls_id, gy, gx] = 1.0
        return self.transform(img), target, loc_target, has_crossing

### Finetuning

In [ ]:
# Folder name of the unzipped dataset under LOCAL_ROOT. Point this at your
# (split) OBB dataset, e.g. "obb_dataset" or "split_dataset".
DATASET_DIRNAME = "split_centroid_dataset"

def _resolve_split(dataset_root):
    """Return the folder holding images/ + labels/ for training.

    Supports a train/val split (use the train/ subfolder) OR a flat dataset
    (images/ and labels/ directly under dataset_root).
    """
    train_split = os.path.join(dataset_root, "train")
    if os.path.isdir(os.path.join(train_split, "images")):
        return train_split
    if os.path.isdir(os.path.join(dataset_root, "images")):
        return dataset_root  # flat layout (no train/val split)
    raise FileNotFoundError(
        f"Could not find images/ under {dataset_root}/train or {dataset_root}."
    )

def run_training():
    # Setup Paths
    dataset_root = os.path.join(LOCAL_ROOT, DATASET_DIRNAME)
    yaml_path = os.path.join(dataset_root, "data.yaml")
    with open(yaml_path, 'r') as f:
        data_cfg = yaml.safe_load(f)

    # Resolve the training split (train/ subfolder, or flat dataset_root).
    train_path = _resolve_split(dataset_root)

    # Set Device to CUDA (NVIDIA GPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")

    num_classes = len(data_cfg['names'])

    # grid_size=30 -> matches the stride-16 (480->30) backbone cut.
    # The loader auto-detects centroid vs YOLO-OBB labels.
    train_ds = YOLOCentroidDataset(train_path, num_classes, grid_size=30)  # hard targets, no smearing
    print(f"Training images: {len(train_ds)} from {train_path}")
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

    model = FOMO_V3_480(num_classes=num_classes).to(device)

    # RetinaNet focal loss with per-class alpha — identical to train_stn_fomo.py.
    class_alpha = build_class_alpha(data_cfg['names']).to(device)
    print(f"Focal alpha: {dict(zip(data_cfg['names'], class_alpha.tolist()))}")
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 60
    for epoch in range(epochs):
        model.train()
        l_sum = 0
        for imgs, targs, loc_targets, has_crossing in train_loader:
            imgs, targs = imgs.to(device), targs.to(device)
            loc_targets  = loc_targets.to(device)   # (B, 2) cos/sin targets
            has_crossing = has_crossing.to(device)   # (B,)   1.0 if crossing present
            optimizer.zero_grad()
            preds, loc_out = model(imgs)  # training=True returns (det, loc)
            if preds.shape[-2:] != targs.shape[-2:]:
                raise RuntimeError(
                    f"Grid mismatch: model output {tuple(preds.shape[-2:])} vs target "
                    f"{tuple(targs.shape[-2:])}. Set grid_size to the model's output resolution."
                )
            det_loss = per_cell_focal_loss(preds, targs, class_alpha)
            # Rotation auxiliary loss: MSE on (cos θ, sin θ), masked to images
            # that actually contain a railroad-crossing (others default to (1,0)
            # which would incorrectly pull the branch toward 0°).
            mask = has_crossing.bool()
            if mask.any():
                rot_loss = F.mse_loss(loc_out[mask], loc_targets[mask])
                loss = det_loss + LOC_WEIGHT * rot_loss
            else:
                loss = det_loss
            loss.backward()
            optimizer.step()
            l_sum += det_loss.item()  # log detection loss only for comparability

        print(f"Epoch {epoch+1}/{epochs} - Det Loss: {l_sum/len(train_loader):.4f}")

    # SAVE TO DRIVE
    save_dir = "/content/drive/MyDrive/fomo_results"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, "fomo_v3_480.pt")
    torch.save(model.state_dict(), save_path)
    print(f"Model saved to: {save_path}")

run_training()